# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

## Load modules

In [13]:
# Module Imports
import selenium                # For navigating borgerforslag.dk
import requests                # For web scraping
import pandas as pd            # For data manipulation and analysis
from bs4 import BeautifulSoup  # For parsing HTML content
import time                    # For managing time delays during scraping
import tqdm                    # For displaying progress bars during scraping
import random                  # For randomizing delays in scraping to avoid detection
import pprint                  # For neatly displaying JSON code
import re                      # For pattern recognition in extracted HTML
from pathlib import Path       # For handling file paths
import csv                     # For exporting data to .csv files
import json                    # For exporting data to .json files
import os                      # For interacting with the operating system (e.g., file paths, environment variables)

# Class and Function Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By                # For using CSS selectors (e.g., cookie-clicking)
from selenium.webdriver.support.ui import WebDriverWait    # For implementing explicit waits
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys            # For simulating keyboard actions (e.g., RETURN key)
from selenium.common.exceptions import NoSuchElementException


## Connecting to Borgerforslag.dk

We first need to connect to Borgerforslag.dk to gather our data. This is accomplished using Selenium. We use the Selenium Chrome Driver to navigate the website, automating tasks such as selecting the correct site path and accepting cookies. Additionally, we adjust the settings to display all proposals, including the expired ones.

In [14]:
# Set Chrome options to disable the search engine choice screen
chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")

# Initialize the Selenium Chrome driver with the specified options
driver = webdriver.Chrome(options=chrome_options)

# URL for Borgerforslag.dk
url_Borgerforslag = "https://borgerforslag.dk/"

# Use the .get() method to open the URL in the Selenium Chrome browser
driver.get(url_Borgerforslag)

# Attempt to locate and click the cookie consent button
try:
    cookie = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'CybotCookiebotDialogBodyLevelButtonLevelOptinAllowallSelection'))
    )
    cookie.click()
except TimeoutException:
    print("Element not found within the specified wait time.")

# Attempt to select "Alle" instead of "Igangværende" proposals
try:
    # Locate and click the filter dropdown
    click_1 = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--value-item'))
    )
    click_1.click()

    # Locate and select the "Alle" option
    click_2 = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--option-0'))
    )
    click_2.click()
except TimeoutException:
    print("Element not found within the specified wait time.")


We have now connected to the site, accepted cookies, and selected all proposals instead of just the active ones. We will now scroll (and click) through the site to be able to view all the elements at once. We create a loop that clicks through this, and stops when no more borgerforslag can be loaded (when the "load more" button disappears).

In [15]:
# Initialize a flag to control the loop
alarm = False  # Defining a stop_alarm

# Loop to navigate through additional pages
while not alarm:
    try:
        # Attempt to locate and click the "load more" button
        click_3 = driver.find_element(By.CSS_SELECTOR, 'button[class="dFsu8t fYY1lZ vFact_DoNotReadAloud _3-IJkM _3CrCss"]')
        click_3.click()

        # Introduce a random delay between clicks to mimic human interaction
        time.sleep(random.uniform(1, 5))
    except NoSuchElementException:
        # If the button is not found, stop the loop
        alarm = True
        print("No more pages to go through")


No more pages to go through


We have now navigated to the bottom of the page in our Selenium Chrome browser, successfully loading all of the borgerforslag. This means we can begin the scraping process, which will collect all of the HTML, including the individual links to each borgerforslag.

In [16]:
from bs4 import BeautifulSoup

# Parse the page source with BeautifulSoup using 'lxml' parser
soup = BeautifulSoup(driver.page_source, 'lxml')

# Find all <a> tags with the specified class
all_sites = soup.find_all('a', class_='lQq327')

# Extract the href attributes from each <a> tag and store them in a list
links = [site['href'] for site in all_sites]

# Print the number of borgerforslag found
print(f"Number of borgerforslag found: {len(links)}\n")

# Print the list of links
for link in links:
    print(link)


Number of borgerforslag found: 1824

https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18153
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18099
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18075
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18046
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18044
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18028
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18016
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17997
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18050
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18032
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18062
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18029
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18030
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17960
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17930
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17914
https://borgerforslag.dk/se-og-stoe

Now that we have all the links to the borgerforslag, we can start scraping the necessary information from each proposal.

In the following block, we define a logging function to record key details about our scraping process for documentation purposes.

In [17]:
# Define the log function to gather and record log information
def log(response, logfile, url, output_path=os.getcwd()):
    # Open or create the log file
    if os.path.isfile(logfile):  # If the log file exists, open it for appending
        log = open(logfile, 'a')
    else:  # If the log file does not exist, create it with headers
        log = open(logfile, 'w')
        header = ['timestamp', 'status_code', 'length', 'url', 'output_file']
        log.write(';'.join(header) + "\n")  # Write headers and move to the next line
        
    # Gather log information
    status_code = response.status_code  # Status code from the HTTP response
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time()))  # Current local time
    length = len(response.text)  # Length of the HTML content
    
    # Append the gathered information to the log file
    with open(logfile, 'a') as log:
        log.write(f'{timestamp};{status_code};{length};{url};{output_path}' + "\n")  # Log the details and move to a new line

With the logging function defined, we can now proceed to scrape data from the individual borgerforslag links.

In [18]:
# Directory to save the log file and output file
output_dir = 'Scraping_files'
os.makedirs(output_dir, exist_ok=True)

# Log file name
log_filename = os.path.join('logfile_borgerforslag.csv')
raw_output_filename = os.path.join(output_dir, 'html_content_raw.txt')
normal_output_filename = os.path.join(output_dir, 'html_content_normal.txt')

# List to store the HTML content from each URL
list_htmls = []

# Limit the range to the first 5 links for testing
test_links = links[:5]

# Loop through each URL in the links list
for i in tqdm.tqdm(links):
    try:
        # Send an HTTP GET request to the current URL with custom headers
        response = requests.get(i, headers={"Name": "Oliver Nyrop Weeks", "Email": "vsn684@alumni.ku.dk"})
        
        # Get the HTML content from the response
        html = response.text
        
        # Append the HTML content to the list
        list_htmls.append(html)
        
        # Log the request details
        log(response, log_filename, i)
        
        # Sleep for 0.5 seconds to avoid overloading the server
        time.sleep(random.uniform(1, 2))
    except Exception as e:
        # If an error occurs, print the URL and the error
        print(f"Error with URL: {i}")
        print(e)
        # Log the error with a status code of 0 (indicating failure)
        log(response, log_filename, i)

 33%|███▎      | 609/1824 [27:16<54:25,  2.69s/it]  


KeyboardInterrupt: 

We have now scraped all the data and stored in a list. We also save it physically:

In [ ]:
with open(raw_output_filename, 'w', encoding='utf-8') as raw_output_file, \
     open(normal_output_filename, 'w', encoding='utf-8') as normal_output_file:
    
    for html_content in list_htmls:
        # Escape newlines and special characters for the raw version
        raw_string = repr(html_content)
        raw_output_file.write(raw_string + "\n")
        
        # Write the normal version without escaping
        normal_output_file.write(html_content + "\n")


In [ ]:
list_htmls

[]

We now have a list and/or a physical file of the html from all the borgerforslag. We now proceed to take out the data from the html:

In [9]:
import pandas as pd
from bs4 import BeautifulSoup

# Assuming list_htmls contains your HTML strings
# List to store the extracted data
extracted_data = []

# Loop through each HTML content and extract the required information
for html_content in list_htmls:
    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, 'lxml')
    
    # Extract the title
    title = soup.title.string if soup.title else "Title not found."
    
    # Initialize variables with default values
    start_date = "Start date not found."
    end_date = "End date not found."
    votes = "Votes not found."
    main_body = "Main body text not found."
    
    # Locate the relevant section that contains the start date, end date, and votes
    date_section = soup.find('div', class_='_3l86Vg')
    
    if date_section:
        date_info = date_section.find_all('div')
        for info in date_info:
            if 'Startdato' in info.text:
                start_date = info.find('strong').text
            if 'Slutdato' in info.text:
                end_date = info.find('strong').text
            if 'Antal støtter' in info.text:
                votes = info.find('strong').text
    
    # Extract the main body text
    main_body_section = soup.find('div', class_='_3dLODA')
    if main_body_section:
        paragraphs = main_body_section.find_all('div', class_='_3mHEzx')
        main_body = "\n".join([p.get_text(strip=True) for p in paragraphs])
    
    # Append the extracted data to the list
    extracted_data.append({
        'Title': title,
        'Start Date': start_date,
        'End Date': end_date,
        'Votes': votes,
        'Main Body': main_body
    })

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(extracted_data)

# Display the DataFrame
df


""


In [10]:
print(df['Main Body'][0:1])

KeyError: 'Main Body'